In [146]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [147]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Preprocessing

In [148]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Encoding

In [149]:
label_encorder_gender=LabelEncoder()
data['Gender']=label_encorder_gender.fit_transform(data['Gender'])
data['Gender']

0       0
1       0
2       0
3       0
4       0
       ..
9995    1
9996    1
9997    0
9998    1
9999    0
Name: Gender, Length: 10000, dtype: int64

In [150]:
one_hot_encoder_geo=OneHotEncoder()
geo_encoder=one_hot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [151]:
geo_encoded_df = pd.DataFrame(
    geo_encoder.toarray(),
    columns=one_hot_encoder_geo.get_feature_names_out(['Geography'])
)
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [152]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [153]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [154]:
with open('label_encorder_gender.pkl','wb') as file:
    pickle.dump(label_encorder_gender,file)

with open('onehot_encorder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

In [155]:
# Divide dataset into independent and dependent features
X = data.drop('Exited', axis=1)
y = data['Exited']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [156]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## ANN implementation

In [157]:
import tensorflow as tf
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [158]:
model=Sequential(
    [
        Dense(64,activation='relu',input_shape=(X_train.shape[1],)),#first hidden layer connected with input layer
        Dense(32,activation='relu'),
        Dense(1,activation='sigmoid')
    ]
)

/opt/anaconda3/envs/venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [159]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [160]:
from tensorflow.keras.optimizers import Adam
opt=Adam(learning_rate=0.01)


In [161]:
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [162]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import os
log_dir = os.path.join(
    "logs",
    "fit",
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [163]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True)


In [164]:
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8300 - loss: 0.4005 - val_accuracy: 0.8415 - val_loss: 0.3592
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 974us/step - accuracy: 0.8549 - loss: 0.3566 - val_accuracy: 0.8620 - val_loss: 0.3476
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 964us/step - accuracy: 0.8560 - loss: 0.3500 - val_accuracy: 0.8610 - val_loss: 0.3404
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - accuracy: 0.8556 - loss: 0.3467 - val_accuracy: 0.8615 - val_loss: 0.3417
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 974us/step - accuracy: 0.8587 - loss: 0.3400 - val_accuracy: 0.8615 - val_loss: 0.3577
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 996us/step - accuracy: 0.8599 - loss: 0.3385 - val_accuracy: 0.8610 - val_loss: 0.3403
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8619 - loss: 0.3367 - val_accuracy: 0.8585 - val_loss: 0.3404
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8605 - loss: 0.3349 

In [165]:
model.save('model.h5')

In [166]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [167]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 9004), started 3:55:29 ago. (Use '!kill 9004' to kill it.)

In [168]:
model=load_model('model.h5')

In [169]:
with open('label_encorder_gender.pkl','rb') as file:
    label_encorder_gender=pickle.load(file)

with open('onehot_encorder_geo.pkl','rb') as file:
    label_encorder_geo=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

In [170]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [171]:
geo_encoded=label_encorder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded


/opt/anaconda3/envs/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[1., 0., 0.]])

In [172]:
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encorder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [173]:
input_df=pd.DataFrame([input_data])
input_data=pd.concat([input_df.reset_index(drop=True),geo_encoded_df],axis=1)

In [174]:
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [175]:
input_data['Gender']=label_encorder_gender.transform(input_df['Gender'])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [176]:
input_data=input_data.drop('Geography',axis=1)
input_data


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [177]:
input_scaled=scaler.transform(input_data)

In [178]:
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [181]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


array([[0.02312229]], dtype=float32)

In [184]:
prediction_proba=prediction[0][0]
prediction_proba

np.float32(0.02312229)

In [185]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
